In [ ]:

# ! pip install -r ../../../../requirements.txt -U

In [ ]:
from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from azure.identity import AzureCliCredential

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)
from azure.core.exceptions import ResourceNotFoundError



In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
AgentName = "Search-Agent"
AgentInstructions = """You are a helpful assistant that can search the web for current information and questions about different kinds of facts.
            **Always** use the Bing search tool whenever asked to provide a piece of information or to validate knowledge. Provide accurate, well-sourced and contextualized answers. 
            Always cite your sources when you use the Bing search results when responding. 
            Add something fun, but based on facts, at the end of your answer."""

In [ ]:
# 1. Resolve the Foundry project connection ID from connection name candidates
env_connection_name = os.environ["BING_CONNECTION_NAME"]

with AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
) as project:
    bing_connection = None
    try:
        bing_connection = project.connections.get(env_connection_name)
        print(f"Resolved connection: {bing_connection.name} -> {bing_connection.id}")
    except ResourceNotFoundError:
        pass

    if bing_connection is None:
        available = [c.name for c in project.connections.list()]
        raise RuntimeError(
            f"No matching Bing connection found. Tried: {env_connection_name}. "
            f"\nAvailable project connections: {available}"
        )

    # 2. Check if agent exists and get latest version
    agent_name = AgentName
    latest_version = None
    
    try:
        # Try to get the latest version of the agent
        versions = list(project.agents.list_versions(agent_name=agent_name))
        if versions:
            latest_agent = versions[0]  # Get the latest version
            latest_version = int(latest_agent.version) if hasattr(latest_agent, 'version') else latest_agent.version
            print(f"Agent '{agent_name}' exists. Latest version: {latest_version}.")
    except ResourceNotFoundError:
        print(f"Agent '{agent_name}' does not exist. A new agent will be created.")

    # 3. Create a new agent version with Bing grounding tool attached (sync path)
    model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME")
    agent = project.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=AgentInstructions,
            tools=[
                BingGroundingTool(
                    bing_grounding=BingGroundingSearchToolParameters(
                        search_configurations=[
                            BingGroundingSearchConfiguration(
                                project_connection_id=bing_connection.id
                            )
                        ]
                    )
                )
            ],
        ),
        description="Bing-grounded search agent",
    )
    
    if int(agent.version) == int(latest_version):
        print(f"No changes detected, agent version remains the same: {agent.version}")
    else:
        print(f"\n✓ Agent version {agent.version} created successfully")
        print(f"  - Agent ID: {agent.id}")
        print(f"  - Agent Name: {agent.name}")
        print(f"  - Version: {agent.version}")
        if latest_version:
            print(f"  - Previous version: {latest_version}")

In [ ]:
UserQuestion = "When was the city of Casablanca founded?"

In [ ]:
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
)

openai_client = project_client.get_openai_client()

# Reference the agent to get a response
response = openai_client.responses.create(
    input=[{"role": "user", "content": UserQuestion}],
    extra_body={"agent_reference": {"name": AgentName, "version": agent.version, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")